# DeepWeeds — Lab Day 2
Mọi lựa chọn dùng validation; chỉ bật test sau khi chốt cấu hình.

In [ ]:
!pip install -q torch torchvision timm fvcore scikit-learn pandas matplotlib openpyxl
import sys
sys.path.insert(0, 'code')
from dataset import load_split, check_split
train_df, val_df, test_df = load_split('data/labels', fold=0)
split_info = check_split(train_df, val_df, test_df, 'data/images')
split_info

## EDA

In [ ]:
import matplotlib.pyplot as plt
from dataset import CLASS_NAMES
counts = train_df.Label.value_counts().reindex(range(9), fill_value=0)
ax = counts.rename(index=dict(enumerate(CLASS_NAMES))).plot.bar(figsize=(11,4), title='Train class distribution')
ax.set_ylabel('Images')
plt.tight_layout()
print('max/min ratio:', counts.max()/counts.min())

## Kiểm tra focal loss và CutMix

In [ ]:
import torch
import torch.nn.functional as F
from losses import FocalLoss, mix_batch
z, y = torch.randn(16,9), torch.randint(0,9,(16,))
assert torch.allclose(FocalLoss(gamma=0)(z,y), F.cross_entropy(z,y), atol=1e-6)
x = torch.randn(8,3,224,224)
xm, (_, _, lam) = mix_batch(x, y[:8], mode='cutmix')
assert xm.shape == x.shape and 0 <= lam <= 1
print('Checks passed; actual CutMix lambda:', lam)

## Bước 1 — backbone (val only)

In [ ]:
from train import Config, run
backbones = ['resnet50','resnext50_32x4d','convnext_tiny','deit_small_patch16_224','efficientnet_b0']
# for i, backbone in enumerate(backbones, 1):
#     run(Config(exp_id=f'B{i:02d}', backbone=backbone, seed=0))

## Bước 2–3 — ablation và inference trên val

In [ ]:
experiments = [Config(exp_id='T00'), Config(exp_id='T01', loss='ls', label_smoothing=0.1), Config(exp_id='T02', mix='cutmix'), Config(exp_id='T03', ema_decay=0.999)]
# for cfg in experiments: run(cfg)

## Bước 4 — chỉ chạy sau khi chốt trên val

In [ ]:
# for seed in (0,1,2):
#     run(Config(exp_id='F01', seed=seed, backbone='BACKBONE_DA_CHOT', save_test_predictions=True))
#     run(Config(exp_id='T00', seed=seed, save_test_predictions=True))

## Chấm cuối
```bash
python eval.py score --pred "predictions/F01_seed*_test.csv" --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv --tag F01 --out eval_out
python eval.py grade --final "predictions/F01_seed*_test.csv" --baseline "predictions/T00_seed*_test.csv" --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv
```